# Helper module function checks
Notebook di smoke test per le funzioni singole in `helper`.
Usa mock locali per evitare dipendenze da rete/Mongo durante i check base.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

In [ ]:
import pandas as pd
import tempfile

from helper.ingest.load_igdb import load_retro_games

sample = pd.DataFrame([
    {"id": 1, "name": "Pac Maze", "summary": "Collect dots", "genres": "['Maze']", "platforms": "['NES']", "game_modes": "['Single player']"},
    {"id": 2, "name": "Modern Racer", "summary": "Racing", "genres": "['Racing']", "platforms": "['PlayStation 5']", "game_modes": "['Single player']"},
])
csv_path = Path(tempfile.gettempdir()) / 'igdb_sample.csv'
sample.to_csv(csv_path, index=False)

retro = load_retro_games(str(csv_path))
assert len(retro) == 1
assert retro.iloc[0]['name'] == 'Pac Maze'
retro

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from unittest.mock import patch

from helper.ingest.download_videos import search_and_download_video

class DummyItem:
    def __init__(self, files):
        self.files = files

def fake_download(identifier, files, destdir, **kwargs):
    target = Path(destdir) / files[0]
    target.write_bytes(b'fake-mp4')

with TemporaryDirectory() as td:
    with patch('helper.ingest.download_videos.ia.search_items', return_value=[{'identifier': 'demo-item'}]), \
         patch('helper.ingest.download_videos.ia.get_item', return_value=DummyItem([{'name': 'play.mp4'}])), \
         patch('helper.ingest.download_videos.ia.download', side_effect=fake_download):
        downloaded = search_and_download_video('Pac Maze', td)

    assert downloaded is not None
    assert Path(downloaded).exists()
    downloaded

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from unittest.mock import patch

from helper.ingest.chunk_videos import chunk_video

class FakeSegment:
    def __init__(self, start, end):
        self.start = start
        self.end = end

    def write_videofile(self, path, logger=None):
        Path(path).write_bytes(b'chunk')

    def close(self):
        return None

class FakeVideoFileClip:
    def __init__(self, *_args, **_kwargs):
        self.duration = 130

    def __enter__(self):
        return self

    def __exit__(self, *_args):
        return None

    def subclip(self, start, end):
        return FakeSegment(start, end)

with TemporaryDirectory() as td:
    source = Path(td) / 'source.mp4'
    source.write_bytes(b'video')
    out = Path(td) / 'chunks'

    with patch('helper.ingest.chunk_videos.VideoFileClip', FakeVideoFileClip):
        chunks = chunk_video(str(source), str(out), chunk_duration=75, overlap=5)

assert len(chunks) == 2
chunks

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from helper.ingest.embed_and_store import embed_text, embed_video, ingest_game

class FakeEmbedding:
    def __init__(self, values):
        self.values = values

class FakeResponse:
    def __init__(self, values):
        self.embeddings = [FakeEmbedding(values)]

class FakeModels:
    def embed_content(self, **_kwargs):
        return FakeResponse([0.1, 0.2, 0.3, 0.4])

class FakeGeminiClient:
    def __init__(self):
        self.models = FakeModels()

class FakeCollection:
    def __init__(self):
        self.docs = []

    def insert_many(self, docs):
        self.docs.extend(docs)

client = FakeGeminiClient()
collection = FakeCollection()

with TemporaryDirectory() as td:
    chunk_file = Path(td) / 'chunk.mp4'
    chunk_file.write_bytes(b'video-bytes')

    game = {
        'id': 99,
        'name': 'Candy Runner',
        'summary': 'Collect sweets and avoid enemies',
        'genres': ['Platformer'],
        'platforms': ['NES'],
        'game_modes': ['Single player'],
    }
    chunks = [{
        'chunk_path': str(chunk_file),
        'start_sec': 0,
        'end_sec': 10,
        'chunk_idx': 0,
    }]

    text_vector = embed_text(client, 'test summary', embed_dim=4)
    video_vector = embed_video(client, str(chunk_file), embed_dim=4)
    inserted = ingest_game(collection, client, game, chunks, embed_dim=4, sleep_seconds=0)

assert len(text_vector) == 4
assert len(video_vector) == 4
assert inserted == 2
assert len(collection.docs) == 2
inserted, collection.docs[0]['chunk_type'], collection.docs[1]['chunk_type']

In [ ]:
from helper.retrieval.query import build_query_text, retrieve_games

class FakeAggregateCollection:
    def __init__(self):
        self.pipeline = None

    def aggregate(self, pipeline):
        self.pipeline = pipeline
        return [{'_id': 99, 'game_name': 'Candy Runner', 'score': 0.98}]

profile = {
    'preferred_genres': ['Platformer', 'Maze'],
    'game_mode': 'Single player',
    'theme_interests': ['candy', 'colorful'],
}

query = build_query_text(profile)
assert 'Platformer' in query

agg_collection = FakeAggregateCollection()
results = retrieve_games(agg_collection, client, profile, top_k=1, embed_dim=4)

assert results[0]['game_name'] == 'Candy Runner'
assert agg_collection.pipeline[0]['$vectorSearch']['filter']['game_modes'] == 'Single player'
results